# MMA3001 – Occupancy Sensor Analysis
Run the eight parts in order in Google Colab after uploading the authorised occupancy-data ZIP. The original dataset is not included in this public repository.
The analysis predicts *sensor-record status*, not occupied duration.


In [ ]:

# PART 1: Import Libraries and Check Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os

# Path to the uploaded dataset
zip_path = "/5occupancySensor_MayToDec2024_9MRows.zip"

# Check file
assert os.path.exists(zip_path), "Dataset not found. Please upload the ZIP file."

with zipfile.ZipFile(zip_path, "r") as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    ]

    assert len(csv_files) > 0, "No CSV file found in ZIP."

    csv_name = csv_files[0]

print("Dataset loaded successfully!")
print("CSV file:", csv_name)

# Preview the first 5 records
with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(csv_name) as f:
        preview = pd.read_csv(f, nrows=5)

display(preview)

print("Columns:", preview.columns.tolist())


In [ ]:

# PART 2: Data Cleaning and Feature Engineering

# Read every 30th record to reduce memory usage
sampled_chunks = []
row_offset = 0

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(csv_name) as f:
        for chunk in pd.read_csv(
            f,
            usecols=["collecteddate", "floorspaceid", "occupancystatus"],
            chunksize=100000
        ):
            positions = np.arange(row_offset, row_offset + len(chunk))
            sampled_chunks.append(
                chunk.iloc[positions % 30 == 0].copy()
            )
            row_offset += len(chunk)

df = pd.concat(sampled_chunks, ignore_index=True)

# Clean data
df["collecteddate"] = pd.to_datetime(
    df["collecteddate"], errors="coerce", utc=True
)

df = df.dropna(
    subset=["collecteddate", "floorspaceid", "occupancystatus"]
)

df = df[df["occupancystatus"].isin([
    "CurrentlyOccupied", "RecentlyOccupied", "NotOccupied"
])].copy()

# Convert to Melbourne time
local_time = df["collecteddate"].dt.tz_convert("Australia/Melbourne")

# Create input features
df["hour"] = local_time.dt.hour
df["day_of_week"] = local_time.dt.dayofweek
df["month"] = local_time.dt.month

# Target: 1 = NotOccupied, 0 = Other Status
df["target"] = (df["occupancystatus"] == "NotOccupied").astype(int)

# Sort by time
df = df.sort_values("collecteddate").reset_index(drop=True)

print("Original records:", row_offset)
print("Sampled and cleaned records:", len(df))
print("\nTarget distribution:")
print(df["target"].value_counts())
print("\nTarget percentage:")
print((df["target"].value_counts(normalize=True) * 100).round(2))

display(df.head())


In [ ]:

# PART 3: Exploratory Data Analysis

import matplotlib.pyplot as plt

# Figure 1: Occupancy Status Distribution
status_counts = df["occupancystatus"].value_counts()

plt.figure(figsize=(8, 4))
status_counts.plot(kind="bar", color="steelblue")
plt.title("Occupancy Status Distribution")
plt.xlabel("Occupancy Status")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("occupancy_distribution.png", dpi=300)
plt.show()

# Figure 2: Hourly Occupancy Record Pattern
df["is_occupied"] = (
    df["occupancystatus"] == "CurrentlyOccupied"
).astype(int)

hourly = df.groupby("hour")["is_occupied"].agg(["mean", "count"])
hourly["percentage"] = hourly["mean"] * 100

plt.figure(figsize=(9, 4))
plt.plot(hourly.index, hourly["percentage"], marker="o")
plt.title("Hourly Occupancy Record Pattern")
plt.xlabel("Hour of Day (Melbourne Time)")
plt.ylabel("CurrentlyOccupied Records (%)")
plt.xticks(range(0, 24, 2))
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("hourly_occupancy.png", dpi=300)
plt.show()

print("EDA completed successfully!")


In [ ]:

# PART 4: Prepare Training and Test Data

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Define features and target
features = ["hour", "day_of_week", "month", "floorspaceid"]

X = df[features]
y = df["target"]

# Split data chronologically
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Define preprocessing (fitted later on training data only)
preprocessor = ColumnTransformer([
    ("time", StandardScaler(), ["hour", "day_of_week", "month"]),
    ("space", OneHotEncoder(handle_unknown="ignore"), ["floorspaceid"])
])

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

print("\nTraining end:", df.iloc[split_index - 1]["collecteddate"])
print("Testing start:", df.iloc[split_index]["collecteddate"])

print("Part 4 completed!")


In [ ]:

# PART 5: Machine Learning and Model Comparison

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)
import pandas as pd

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Logistic Regression model
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])
model.fit(X_train, y_train)

# Make predictions
baseline_pred = baseline.predict(X_test)
lr_pred = model.predict(X_test)

# Evaluation function
def evaluate(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

# Compare models
comparison = pd.DataFrame([
    {"Model": "Baseline", **evaluate(y_test, baseline_pred)},
    {"Model": "Logistic Regression", **evaluate(y_test, lr_pred)}
])

print("Model Comparison:")
display(comparison.round(4))

comparison.to_csv("model_comparison.csv", index=False)
print("Part 5 completed!")


In [ ]:

# PART 6: Threshold Optimisation

from sklearn.base import clone
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

# Split training data into fitting and validation sets
split = int(len(X_train) * 0.8)

X_fit = X_train.iloc[:split]
X_val = X_train.iloc[split:]
y_fit = y_train.iloc[:split]
y_val = y_train.iloc[split:]

# Train a separate model for threshold selection
val_model = clone(model)
val_model.fit(X_fit, y_fit)

# Predict probabilities on validation data
val_prob = val_model.predict_proba(X_val)[:, 1]

# Test different thresholds
thresholds = np.arange(0.1, 0.91, 0.05)

scores = []

for threshold in thresholds:
    predictions = (val_prob >= threshold).astype(int)
    scores.append(f1_score(y_val, predictions, zero_division=0))

# Select threshold with highest validation F1
best_index = np.argmax(scores)
best_threshold = float(thresholds[best_index])

print("Best threshold:", round(best_threshold, 2))
print("Validation F1:", round(scores[best_index], 4))

# Evaluate optimised model on independent test data
test_prob = model.predict_proba(X_test)[:, 1]
optimised_pred = (test_prob >= best_threshold).astype(int)

# Final comparison
final_comparison = pd.DataFrame([
    {"Model": "Baseline", **evaluate(y_test, baseline_pred)},
    {"Model": "Logistic Regression (0.50)", **evaluate(y_test, lr_pred)},
    {"Model": "Optimised Logistic Regression",
     **evaluate(y_test, optimised_pred)}
])

print("\nFinal Model Comparison:")
display(final_comparison.round(4))

final_comparison.to_csv("final_model_comparison.csv", index=False)

print("Part 6 completed!")


In [ ]:

# PART 7: Verification and Performance

import time
import numpy as np
import pandas as pd
from sklearn.base import clone

# 1. Test target encoding
statuses = pd.Series([
    "CurrentlyOccupied",
    "NotOccupied",
    "RecentlyOccupied"
])
assert (statuses == "NotOccupied").astype(int).tolist() == [0, 1, 0]

# 2. Test time conversion
test_date = pd.Timestamp("2024-06-01", tz="UTC")
assert test_date.tz_convert("Australia/Melbourne").hour == 10

# 3. Test model probabilities
probabilities = model.predict_proba(X_test.iloc[:10])
assert probabilities.shape == (10, 2)
assert np.allclose(probabilities.sum(axis=1), 1)
assert np.all((probabilities >= 0) & (probabilities <= 1))

# 4. Verify chronological splitting
assert df.iloc[split_index - 1]["collecteddate"] <= df.iloc[split_index]["collecteddate"]

print("All 4 verification tests passed!")

# 5. Measure computational performance
performance = []

for name, estimator in [
    ("Baseline", baseline),
    ("Logistic Regression", model)
]:
    estimator_copy = clone(estimator)

    start = time.perf_counter()
    estimator_copy.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    estimator_copy.predict(X_test)
    predict_time = time.perf_counter() - start

    performance.append({
        "Model": name,
        "Training_Time_s": train_time,
        "Prediction_Time_s": predict_time
    })

performance_df = pd.DataFrame(performance)

print("\nComputational Performance:")
display(performance_df.round(4))

performance_df.to_csv("computational_performance.csv", index=False)

print("Part 7 completed!")


In [ ]:

# PART 8: Final Results and Export

import os
import shutil
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Create results folder
os.makedirs("results", exist_ok=True)

# 1. Model comparison chart
metrics = ["Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1"]

ax = final_comparison.set_index("Model")[metrics].T.plot(
    kind="bar",
    figsize=(11, 5)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("results/model_comparison.png", dpi=300)
plt.show()

# 2. Optimised model confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test,
    optimised_pred,
    display_labels=["Other Status", "NotOccupied"],
    cmap="Blues",
    values_format="d"
)

plt.title("Optimised Model Confusion Matrix")
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=300)
plt.show()

# 3. Copy other figures and results
files_to_copy = [
    "occupancy_distribution.png",
    "hourly_occupancy.png",
    "model_comparison.csv",
    "final_model_comparison.csv",
    "computational_performance.csv"
]

for filename in files_to_copy:
    if os.path.exists(filename):
        shutil.copy2(filename, os.path.join("results", filename))
    else:
        print("Missing file:", filename)

# 4. Save a simple summary
with open("results/summary.txt", "w") as f:
    f.write("MMA3001 Occupancy Analysis\n")
    f.write(f"Dataset sample: {len(df)} records\n")
    f.write(f"Best threshold: {best_threshold:.2f}\n")
    f.write("\nFinal Model Comparison:\n")
    f.write(final_comparison.to_string(index=False))
    f.write("\n\nComputational Performance:\n")
    f.write(performance_df.to_string(index=False))

# 5. Create ZIP of result files
shutil.make_archive("MMA3001_results", "zip", "results")

print("Results exported successfully!")
print("Files:", os.listdir("results"))
print("Part 8 completed!")
